In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)

CONTRAST_ORDER = ["RBD vs HC", "Hyposmia vs HC"]

MAP_INFO = {
    "Serotonin | target-5HT1a_tracer-way100635_n-35_dx-hc_pub-savli2012":       ("5-HT1a",  "Serotonin"),
    "Serotonin | target-5HT1b_tracer-p943_n-23_dx-hc_pub-savli2012":            ("5-HT1b",  "Serotonin"),
    "Serotonin | target-5HT2a_tracer-altanserin_n-19_dx-hc_pub-savli2012":      ("5-HT2a",  "Serotonin"),
    "Serotonin | target-5HT4_tracer-sb207145_n-59_dx-hc_pub-beliveau2017":      ("5-HT4",   "Serotonin"),
    "Serotonin | target-5HTT_tracer-dasb_n-18_dx-hc_pub-savli2012":             ("5-HTT",   "Serotonin"),
    "Dopamine | target-D1_tracer-sch23390_n-13_dx-hc_pub-kaller2017":           ("D1",      "Dopamine"),
    "Dopamine | target-D23_tracer-flb457_n-55_dx-hc_pub-sandiego2015":          ("D2/3",    "Dopamine"),
    "Dopamine | target-DAT_tracer-fpcit_n-174_dx-hc_pub-dukart2018":            ("DAT",     "Dopamine"),
    "Dopamine | target-FDOPA_tracer-fluorodopa_n-12_dx-hc_pub-garciagomez2018": ("FDOPA",   "Dopamine"),
    "GABA | target-GABAa_tracer-flumazenil_n-6_dx-hc_pub-dukart2018":           ("GABAa",   "GABA"),
    "Glutamate | target-mGluR5_tracer-abp688_n-73_dx-hc_pub-smart2019":         ("mGluR5",  "Glutamate"),
    "Glutamate | target-NMDA_tracer-ge179_n-29_dx-hc_pub-galovic2021":          ("NMDA",    "Glutamate"),
    "Noradrenaline/Acetylcholine | target-NET_tracer-mrb_n-10_dx-hc_pub-hesse2017":           ("NET",    "NA/ACh"),
    "Noradrenaline/Acetylcholine | target-VAChT_tracer-feobv_n-18_dx-hc_pub-aghourian2017":   ("VAChT",  "NA/ACh"),
}

SYSTEM_COLORS = {
    "Serotonin":  "#4E9AC7",
    "Dopamine":   "#E07B54",
    "GABA":       "#9B6CB0",
    "Glutamate":  "#5FAD71",
    "NA/ACh":     "#E0A0C0",
}

SYSTEM_BANDS = [
    ( 0,  4, "#f0f5fc"),
    ( 5,  8, "#fdf3ee"),
    ( 9,  9, "#f5f0fc"),
    (10, 11, "#eefaf2"),
    (12, 13, "#fceef5"),
]

MAP_ORDER = list(MAP_INFO.keys())

In [ ]:
OUT_DIR = Path("../../results/figures/lollipop")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def match_full_map(ref_str):
    for full in MAP_ORDER:
        key = full.split("target-")[-1].split("_tracer")[0].lower()
        if key in ref_str.lower():
            return full
    return None


def build_lookups(csv_path):
    df = pd.read_csv(csv_path)
    df["map_full"] = df["reference_map"].apply(match_full_map)
    df = df.dropna(subset=["map_full"])
    rho_lk = {(r["map_full"], r["contrast"]): r["rho"] for _, r in df.iterrows()}
    q_lk   = {(r["map_full"], r["contrast"]): r["q"]   for _, r in df.iterrows()}
    return rho_lk, q_lk


rho_thick, q_thick = build_lookups("../../results/nispace_group_comparison_results_subgroups_thickness.csv")
rho_subc,  q_subc  = build_lookups("../../results/nispace_group_comparison_results_subgroups_subcortical.csv")

print(f"Thickness   — {len(rho_thick)} entries")
print(f"Subcortical — {len(rho_subc)} entries")

In [ ]:
def sig_star(q):
    if q < 0.001: return "***"
    if q < 0.01:  return "**"
    if q < 0.05:  return "*"
    return ""


def draw_lollipop_panel(ax, contrast, rho_lookup, q_lookup, map_order, map_info,
                        system_colors, system_bands, show_ylabels=False):
    n = len(map_order)
    y_pos = np.arange(n - 1, -1, -1)

    system_band_positions = [(0, 4), (5, 8), (9, 9), (10, 11), (12, 13)]
    for (mi0, mi1), (_, _, col) in zip(system_band_positions, system_bands):
        y_hi = (n - 1 - mi0) + 0.5
        y_lo = (n - 1 - mi1) - 0.5
        ax.axhspan(y_lo, y_hi, color=col, alpha=1.0, zorder=0, lw=0)

    for b in [4.5, 8.5, 9.5, 11.5]:
        ax.axhline((n - 1) - b, color="#cccccc", lw=0.8, zorder=1)

    ax.axvline(0, color="#888888", lw=0.8, linestyle="--", alpha=0.6, zorder=1)

    for yi, full_map in zip(y_pos, map_order):
        short, system = map_info[full_map]
        rho = rho_lookup.get((full_map, contrast), 0.0)
        q   = q_lookup.get((full_map, contrast), 1.0)
        is_sig = q < 0.05
        color  = system_colors[system]

        if is_sig:
            ax.plot([0, rho], [yi, yi], color=color, lw=2.5, zorder=2)
            ax.scatter([rho], [yi], s=90, color=color, edgecolors="white",
                       linewidths=1.5, zorder=3)
            star = sig_star(q)
            if star:
                offset = 0.055 if rho >= 0 else -0.055
                ha = "left" if rho >= 0 else "right"
                ax.text(rho + offset, yi, star, ha=ha, va="center",
                        fontsize=10, fontweight="bold", color="black")
        else:
            ax.plot([0, rho], [yi, yi], color="#aaaaaa", lw=1.2, alpha=0.5, zorder=2)
            ax.scatter([rho], [yi], s=50, facecolors="white",
                       edgecolors="#aaaaaa", linewidths=1.2, zorder=3)

        if show_ylabels:
            ax.text(-1.18, yi, short, ha="right", va="center",
                    fontsize=9, fontweight="semibold", color=color, clip_on=False)

    ax.set_xlim(-1.05, 1.05)
    ax.set_xticks(np.arange(-1.0, 1.01, 0.5))
    ax.set_ylim(-0.7, n - 0.3)
    ax.set_xlabel("Spearman ρ", fontsize=9.5)
    ax.set_title(contrast, fontsize=10, fontweight="bold", loc="center", pad=8)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", which="major", alpha=0.2, lw=0.7)
    ax.set_yticks([])
    ax.tick_params(axis="y", left=False, labelleft=False)

In [ ]:
# ── Cortical thickness lollipop ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(9.5, 7), sharey=True)

fig.suptitle(
    "Group-level colocalization: Cortical Thickness (Prodromal Subgroups)",
    fontsize=13, fontweight="bold", y=1.02
)

for i, (contrast, ax) in enumerate(zip(CONTRAST_ORDER, axes)):
    draw_lollipop_panel(
        ax, contrast, rho_thick, q_thick,
        MAP_ORDER, MAP_INFO, SYSTEM_COLORS, SYSTEM_BANDS,
        show_ylabels=(i == 0)
    )
    ax.text(-0.05, 1.05, "abcdefghijklmnopqrstuvwxyz"[i],
            transform=ax.transAxes, fontsize=12,
            fontweight='bold', va='top', clip_on=False)

legend_patches = [
    mpatches.Patch(facecolor=c, alpha=0.75, label=s)
    for s, c in SYSTEM_COLORS.items()
]
fig.legend(
    handles=legend_patches, loc="lower center", ncol=5,
    frameon=False, fontsize=9, bbox_to_anchor=(0.5, 0.02),
)
fig.text(
    0.5, 0.01,
    "Filled dot = FDR q < 0.05 (group-level); open dot = non-significant.\n"
    "ρ = group-level Spearman correlation between cortical thickness deviation map and neurotransmitter reference map.",
    ha="center", va="top", fontsize=9.5, color="#444444", style="italic"
)

plt.subplots_adjust(left=0.14, right=0.98, top=0.90, bottom=0.16, wspace=0.08)

out_path = OUT_DIR / "figure_lollipop_colocalization_thickness_subgroups.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

In [ ]:
# ── Subcortical volume lollipop ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(9.5, 7), sharey=True)

fig.suptitle(
    "Group-level colocalization: Subcortical Volume (Prodromal Subgroups)",
    fontsize=13, fontweight="bold", y=1.02
)

for i, (contrast, ax) in enumerate(zip(CONTRAST_ORDER, axes)):
    draw_lollipop_panel(
        ax, contrast, rho_subc, q_subc,
        MAP_ORDER, MAP_INFO, SYSTEM_COLORS, SYSTEM_BANDS,
        show_ylabels=(i == 0)
    )
    ax.text(-0.05, 1.05, "abcdefghijklmnopqrstuvwxyz"[i],
            transform=ax.transAxes, fontsize=12,
            fontweight='bold', va='top', clip_on=False)

legend_patches = [
    mpatches.Patch(facecolor=c, alpha=0.75, label=s)
    for s, c in SYSTEM_COLORS.items()
]
fig.legend(
    handles=legend_patches, loc="lower center", ncol=5,
    frameon=False, fontsize=9, bbox_to_anchor=(0.5, 0.02),
)
fig.text(
    0.5, 0.01,
    "Filled dot = FDR q < 0.05 (group-level); open dot = non-significant.\n"
    "ρ = group-level Spearman correlation between subcortical volume deviation map and neurotransmitter reference map.",
    ha="center", va="top", fontsize=9.5, color="#444444", style="italic"
)

plt.subplots_adjust(left=0.14, right=0.98, top=0.90, bottom=0.16, wspace=0.08)

out_path = OUT_DIR / "figure_lollipop_colocalization_subcortical_subgroups.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()